# OpenScope Predictive Processing: random calcium-imaging NWB + stimulus-triggered response

This Colab-ready notebook:

1. connects to the **OpenScope Predictive Processing mesoscope Dandiset (`001768`)**;
2. reproducibly chooses **one NWB calcium-imaging asset at random** (seeded so the notebook remains reproducible);
3. streams the NWB from DANDI rather than downloading the entire file;
4. plots a representative field-of-view image;
5. finds a ΔF/F calcium trace and the stimulus table;
6. computes a **PSTH-style peri-stimulus time average** (trial-aligned ΔF/F; for calcium imaging this is more precisely a stimulus-triggered average than a spike histogram);
7. plots single-trial responses plus mean ± SEM.

The code intentionally discovers NWB object names instead of hard-coding a single plane name, because object names can vary between sessions.

**Data:** OpenScope Community Predictive Processing  
**Mesoscope Dandiset:** DANDI:001768  
**NWB:** Neurodata Without Borders

> Runtime note: the NWB is streamed with HTTP range requests. The first access to a large dataset can take a little while in Colab.

In [ ]:
# Colab setup
%pip -q install "dandi>=0.70" pynwb h5py remfile pandas scipy matplotlib

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py, remfile

from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

DANDISET_ID = "001768"
DANDISET_VERSION = "draft"
RANDOM_SEED = 20260905   # change this to choose another random file
PRE_S = 1.0
POST_S = 2.0
N_TIMEPOINTS = 151

rng = random.Random(RANDOM_SEED)

## 1. Pick one calcium-imaging NWB at random

We query DANDI live, collect `.nwb` assets, and choose one with a fixed random seed. Printing the selected path makes the analysis fully traceable.

In [ ]:
client = DandiAPIClient()
dandiset = client.get_dandiset(DANDISET_ID, version_id=DANDISET_VERSION)

assets = [a for a in dandiset.get_assets() if a.path.endswith(".nwb")]
if not assets:
    raise RuntimeError(f"No NWB assets found in DANDI:{DANDISET_ID}/{DANDISET_VERSION}")

asset = rng.choice(assets)
print(f"Found {len(assets)} NWB assets")
print("Randomly selected:")
print(" ", asset.path)
print("size:", getattr(asset, "size", "unknown"))

## 2. Stream the selected NWB

`remfile` makes the remote file seekable, so `h5py`/PyNWB request only the byte ranges needed for the analysis.

In [ ]:
url = asset.get_content_url(follow_redirects=1, strip_query=True)
remote_file = remfile.File(url)
h5_file = h5py.File(remote_file, "r")
io = NWBHDF5IO(file=h5_file, mode="r", load_namespaces=True)
nwb = io.read()

print("identifier:", nwb.identifier)
print("session:", nwb.session_description)
print("start:", nwb.session_start_time)
print("subject:", getattr(nwb.subject, "subject_id", None))
print("processing modules:", list(nwb.processing.keys()))
print("interval tables:", list(nwb.intervals.keys()))

## 3. Discover a field-of-view image

Current mesoscope files generally place projection images in an `images` interface within a plane-specific processing module. This helper searches all processing modules and falls back to image-like acquisition objects if needed.

In [ ]:
def iter_images(nwbfile):
    # Processing modules
    for module_name, module in nwbfile.processing.items():
        for interface_name, interface in module.data_interfaces.items():
            # Images container
            if hasattr(interface, "images"):
                try:
                    for image_name, image in interface.images.items():
                        yield f"processing/{module_name}/{interface_name}/{image_name}", image
                except Exception:
                    pass
            # ImageSeries-like interface
            if hasattr(interface, "data") and any(
                k in interface_name.lower() for k in ["image", "projection", "mean", "average", "max"]
            ):
                yield f"processing/{module_name}/{interface_name}", interface

    # Acquisition fallback
    for name, obj in nwbfile.acquisition.items():
        if hasattr(obj, "data") and any(k in name.lower() for k in ["image", "projection", "mean", "average", "max"]):
            yield f"acquisition/{name}", obj

image_candidates = list(iter_images(nwb))
print("Image candidates:")
for p, _ in image_candidates[:20]:
    print(" ", p)

if not image_candidates:
    raise RuntimeError("No image-like NWB object found. Inspect `nwb.processing` for this session.")

# Prefer average/max/mean projection names
def image_score(item):
    p = item[0].lower()
    return sum(k in p for k in ["average", "mean", "max", "projection"])

image_path, image_obj = max(image_candidates, key=image_score)
arr = np.asarray(image_obj.data)

# Reduce singleton/time/channel dimensions until 2-D where possible.
arr = np.squeeze(arr)
while arr.ndim > 2:
    arr = arr[0]
if arr.ndim != 2:
    raise RuntimeError(f"Selected image has unexpected shape {arr.shape}: {image_path}")

lo, hi = np.nanpercentile(arr, [1, 99.5])
plt.figure(figsize=(8, 6))
plt.imshow(arr, cmap="gray", vmin=lo, vmax=hi)
plt.title(f"Field-of-view image\n{image_path}")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Plotted:", image_path, "shape:", arr.shape)

## 4. Find a ΔF/F series

The release documentation describes plane-specific `dff_timeseries` objects. The helper below searches recursively through processing interfaces and returns ROI response series whose names/descriptions look like ΔF/F.

In [ ]:
def collect_timeseries(nwbfile):
    found = []

    def add(path, obj):
        if hasattr(obj, "data") and (hasattr(obj, "timestamps") or hasattr(obj, "rate")):
            found.append((path, obj))

    for module_name, module in nwbfile.processing.items():
        for interface_name, interface in module.data_interfaces.items():
            add(f"processing/{module_name}/{interface_name}", interface)

            # Common NWB containers: DfOverF / Fluorescence
            if hasattr(interface, "roi_response_series"):
                try:
                    for series_name, series in interface.roi_response_series.items():
                        add(f"processing/{module_name}/{interface_name}/{series_name}", series)
                except Exception:
                    pass

    return found

series_candidates = collect_timeseries(nwb)

def dff_score(item):
    path, obj = item
    text = " ".join([
        path,
        str(getattr(obj, "name", "")),
        str(getattr(obj, "description", "")),
        str(getattr(obj, "unit", "")),
    ]).lower()
    score = 0
    for key, pts in [("dff", 8), ("df/f", 8), ("deltaf", 6), ("fluorescence", 2)]:
        if key in text:
            score += pts
    return score

ranked = sorted(series_candidates, key=dff_score, reverse=True)
print("Top time-series candidates:")
for p, obj in ranked[:15]:
    try:
        shape = obj.data.shape
    except Exception:
        shape = "?"
    print(f"  score={dff_score((p,obj)):2d}  shape={shape}  {p}")

if not ranked or dff_score(ranked[0]) == 0:
    raise RuntimeError("Could not identify a ΔF/F-like series automatically.")

dff_path, dff = ranked[0]
print("\nUsing:", dff_path)
print("shape:", dff.data.shape)

In [ ]:
def timestamps_for_series(ts):
    if getattr(ts, "timestamps", None) is not None:
        return np.asarray(ts.timestamps[:], dtype=float)
    if getattr(ts, "rate", None) is not None and getattr(ts, "starting_time", None) is not None:
        n = ts.data.shape[0]
        return float(ts.starting_time) + np.arange(n) / float(ts.rate)
    raise RuntimeError("Series has neither timestamps nor rate/starting_time.")

t = timestamps_for_series(dff)

# Determine time axis. NWB RoiResponseSeries is usually time x ROI.
shape = dff.data.shape
if len(shape) == 1:
    roi_trace = np.asarray(dff.data[:], dtype=float)
    roi_index = None
elif len(shape) == 2:
    if shape[0] == len(t):
        # Choose the ROI with largest variance from a small deterministic sample.
        candidate_idx = np.linspace(0, shape[1]-1, min(shape[1], 30), dtype=int)
        sample = np.asarray(dff.data[:, candidate_idx], dtype=float)
        variances = np.nanvar(sample, axis=0)
        roi_index = int(candidate_idx[np.nanargmax(variances)])
        roi_trace = np.asarray(dff.data[:, roi_index], dtype=float)
    elif shape[1] == len(t):
        candidate_idx = np.linspace(0, shape[0]-1, min(shape[0], 30), dtype=int)
        sample = np.asarray(dff.data[candidate_idx, :], dtype=float)
        variances = np.nanvar(sample, axis=1)
        roi_index = int(candidate_idx[np.nanargmax(variances)])
        roi_trace = np.asarray(dff.data[roi_index, :], dtype=float)
    else:
        raise RuntimeError(f"Cannot match timestamps ({len(t)}) to data shape {shape}")
else:
    raise RuntimeError(f"Expected 1-D or 2-D ΔF/F data, got {shape}")

print("ROI:", roi_index)
print("trace points:", len(roi_trace), "time span:", (t[0], t[-1]))

## 5. Find stimulus onset times

We score interval tables by names/columns associated with visual stimulus presentations. The selected table and its columns are printed before analysis so you can verify that the alignment is scientifically appropriate for the chosen session.

In [ ]:
interval_candidates = []
for name, table in nwb.intervals.items():
    cols = list(table.colnames)
    text = (name + " " + " ".join(cols)).lower()
    score = 0
    if "stim" in text: score += 5
    if "visual" in text: score += 3
    if "orientation" in text: score += 3
    if "trial" in text: score += 1
    if "start_time" in cols: score += 2
    interval_candidates.append((score, name, table))

interval_candidates.sort(key=lambda x: x[0], reverse=True)
print("Interval candidates:")
for score, name, table in interval_candidates:
    print(score, name, list(table.colnames))

if not interval_candidates:
    raise RuntimeError("No NWB interval tables found.")

_, stim_name, stim_table = interval_candidates[0]
stim_df = stim_table.to_dataframe()
print("\nUsing table:", stim_name)
display(stim_df.head())
print("n rows:", len(stim_df))

### Choose a repeated stimulus condition

If an orientation column is present, the notebook automatically chooses the most frequent finite orientation. This gives a conventional stimulus-triggered response for one repeated visual condition. Otherwise it uses all valid stimulus starts.

In [ ]:
events = stim_df.copy()

orientation_col = next((c for c in events.columns if "orientation" in c.lower()), None)
if orientation_col is not None:
    vals = pd.to_numeric(events[orientation_col], errors="coerce")
    finite = vals[np.isfinite(vals)]
    if len(finite):
        chosen_orientation = finite.value_counts().index[0]
        events = events[np.isclose(vals, chosen_orientation, equal_nan=False)]
        condition_label = f"{orientation_col} = {chosen_orientation:g}"
    else:
        condition_label = "all stimuli"
else:
    condition_label = "all stimuli"

onsets = pd.to_numeric(events["start_time"], errors="coerce").to_numpy()
onsets = onsets[np.isfinite(onsets)]
onsets = onsets[(onsets >= t[0] + PRE_S) & (onsets <= t[-1] - POST_S)]

print("Condition:", condition_label)
print("Usable stimulus onsets:", len(onsets))
if len(onsets) < 3:
    raise RuntimeError("Too few usable stimulus events for a peri-stimulus average.")

## 6. PSTH-style stimulus-triggered calcium response

For spike trains, a PSTH bins spike counts. For calcium imaging, the analogous plot is a **peri-stimulus ΔF/F average**. Each trial is interpolated onto a common time grid, baseline-subtracted using the pre-stimulus period, and then averaged.

The first panel shows individual trials. The second shows mean ± SEM.

In [ ]:
rel_t = np.linspace(-PRE_S, POST_S, N_TIMEPOINTS)
trials = []

for onset in onsets:
    y = np.interp(onset + rel_t, t, roi_trace)
    baseline = np.nanmean(y[rel_t < 0])
    trials.append(y - baseline)

trials = np.asarray(trials)
good = np.isfinite(trials).mean(axis=1) > 0.95
trials = trials[good]

mean_resp = np.nanmean(trials, axis=0)
sem_resp = np.nanstd(trials, axis=0, ddof=1) / np.sqrt(trials.shape[0])

fig, ax = plt.subplots(figsize=(9, 5))
for y in trials[:min(100, len(trials))]:
    ax.plot(rel_t, y, alpha=0.08, linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=1)
ax.set(
    xlabel="Time from stimulus onset (s)",
    ylabel="Baseline-subtracted ΔF/F",
    title=f"Single-trial stimulus-triggered calcium responses\n{condition_label} | ROI {roi_index}"
)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rel_t, mean_resp, linewidth=2, label=f"mean (n={len(trials)})")
ax.fill_between(rel_t, mean_resp-sem_resp, mean_resp+sem_resp, alpha=0.25, label="SEM")
ax.axvline(0, linestyle="--", linewidth=1, label="stimulus onset")
ax.set(
    xlabel="Time from stimulus onset (s)",
    ylabel="Baseline-subtracted ΔF/F",
    title=f"PSTH-style peri-stimulus calcium response\n{condition_label} | ROI {roi_index}"
)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Optional: population-average response

If the selected ΔF/F series contains multiple ROIs and time is the first dimension, this cell computes a population trace from up to 100 ROIs and repeats the same alignment.

In [ ]:
if len(dff.data.shape) == 2 and dff.data.shape[0] == len(t):
    n_rois = dff.data.shape[1]
    idx = np.linspace(0, n_rois - 1, min(n_rois, 100), dtype=int)
    population_trace = np.nanmean(np.asarray(dff.data[:, idx], dtype=float), axis=1)

    pop_trials = []
    for onset in onsets:
        y = np.interp(onset + rel_t, t, population_trace)
        y -= np.nanmean(y[rel_t < 0])
        pop_trials.append(y)
    pop_trials = np.asarray(pop_trials)

    pop_mean = np.nanmean(pop_trials, axis=0)
    pop_sem = np.nanstd(pop_trials, axis=0, ddof=1) / np.sqrt(len(pop_trials))

    plt.figure(figsize=(9,5))
    plt.plot(rel_t, pop_mean, linewidth=2)
    plt.fill_between(rel_t, pop_mean-pop_sem, pop_mean+pop_sem, alpha=.25)
    plt.axvline(0, linestyle="--")
    plt.xlabel("Time from stimulus onset (s)")
    plt.ylabel("Population baseline-subtracted ΔF/F")
    plt.title(f"Population stimulus-triggered response ({len(idx)} ROIs)\n{condition_label}")
    plt.tight_layout()
    plt.show()
else:
    print("Population cell skipped for this series shape.")

## 8. Reproducibility record

Run this cell before committing the notebook to GitHub. It records the exact randomly selected DANDI asset and key analysis choices.

In [ ]:
print("Dandiset:", DANDISET_ID)
print("Version:", DANDISET_VERSION)
print("Random seed:", RANDOM_SEED)
print("Asset:", asset.path)
print("NWB identifier:", nwb.identifier)
print("Image:", image_path)
print("ΔF/F series:", dff_path)
print("ROI:", roi_index)
print("Stimulus table:", stim_name)
print("Condition:", condition_label)
print("Trials:", len(trials))
print("Window:", (-PRE_S, POST_S), "seconds")

# Close remote handles when finished.
# io.close()
# h5_file.close()
# remote_file.close()

## Notes for GitHub / Colab

- Commit this `.ipynb` directly to your repository; GitHub renders notebooks natively.
- To make the random choice immutable for a publication/analysis, run the notebook once, copy the printed `asset.path`, and replace the random-selection cell with `dandiset.get_asset_by_path("...")`.
- `draft` follows the current DANDI release. For a frozen analysis, change `DANDISET_VERSION` to a published version once available.
- The automatic NWB discovery is deliberately defensive. Always inspect the printed stimulus table, ΔF/F path, image path, and session metadata before interpreting the result.